<a href="https://colab.research.google.com/github/semanticClimate/semantic_corpus/blob/shreya/Copy_of_corpus_query_review_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Corpus query, download, and review

**Date:** 2026-07-03 (system date)

This notebook helps you:

1. Search Europe PMC for scientific papers
2. Download fulltext XML where available
3. Build a scored review table for human curation
4. Refine your query and compare results

Documentation:
- [build_review_table.md](https://github.com/semanticClimate/semantic_corpus/blob/main/docs/build_review_table.md)
- [aqi_india_corpus_workflow.md](https://github.com/semanticClimate/semantic_corpus/blob/main/docs/aqi_india_corpus_workflow.md)
- [colab_corpus_review_notebook.md](https://github.com/semanticClimate/semantic_corpus/blob/main/docs/colab_corpus_review_notebook.md)

## 1. Install dependencies

In [ ]:
!pip install -q git+https://github.com/semanticClimate/semantic_corpus.git
!pip install -q pandas ipywidgets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 90.9 MB/s eta 0:00:00


## 2. Parameters

Edit these values before each run. Set `REVISION_OF` when refining a previous query.

In [ ]:
from pathlib import Path

QUERY_NAME = "global_warming"
QUERY_STRING = (
    '("global warming" OR "climate change" '
    'OR "greenhouse effect" '
    'OR "global temperature rise")'
)
LIMIT = 50
FORMATS = ["xml"]
REPOSITORY = "europe_pmc"
NOTES = ""
REVISION_OF = None  # e.g. "aqi_india_pilot" when refining a prior run

OUTPUT_DIR = Path("/content/queries", QUERY_NAME)
OUTPUT_DIR

PosixPath('/content/queries/global_warming')

## 3. Run query, download, and build review table

In [ ]:
from semantic_corpus.corpus_review.workflow import run_query_and_build_review_table

result = run_query_and_build_review_table(
    query_name=QUERY_NAME,
    query_string=QUERY_STRING,
    output_dir=OUTPUT_DIR,
    repository=REPOSITORY,
    limit=LIMIT,
    formats=FORMATS,
    notes=NOTES,
    revision_of=REVISION_OF,
)

print(result["summary"])
print(f"Rows: {result['row_count']} (score >= 5: {result['high_score_count']}, with XML: {result['xml_count']})")
print("Review files:")
for fmt, path in result["review_paths"].items():
    print(f"  {fmt}: {path}")

Query 'global_warming': 50 results, 24 downloaded -> /content/queries/global_warming
Rows: 50 (score >= 5: 0, with XML: 24)
Review files:
  json: /content/queries/global_warming/review/review_table.json
  csv: /content/queries/global_warming/review/review_table.csv
  markdown: /content/queries/global_warming/review/review_table.md


## 4. Browse results (immediate reaction)

Inspect top candidates. Score ranks papers for review; it does **not** auto-include them.

In [ ]:
from semantic_corpus.corpus_review.workflow import review_rows_to_dataframe

df = review_rows_to_dataframe(result["rows"])

display_cols = [
    "score",
    "title",
    "pmcid",
    "has_xml",
    "location_terms",
    "pollutant_terms",
    "review_status",
]
display(df[display_cols].head(15))

print("\nHigh-score papers with XML:")
high_xml = df[(df["score"] >= 5) & (df["has_xml"] == True)]
print(f"  {len(high_xml)} of {len(df)} rows")
display(high_xml[display_cols])

,score,title,pmcid,has_xml,location_terms,pollutant_terms,review_status
0,2,The boiling frog effect: Global warming delays...,,False,,air pollution,review
1,2,Benchmarking Regional Climate Variability in C...,,False,"india, indian",,review
2,-1,A rapidly closing window for coral persistence...,PMC12589577,True,,,review
3,-1,Neotropical ants are at greater risk from glob...,PMC13172437,True,,,review
4,-1,The relationship between environmental literac...,PMC13285055,True,,,review
5,-2,Global warming impacts on lactating mammalian ...,,False,,,review
6,-2,Global Warming and the Elderly: A Socio-Ecolog...,,False,,,review
7,-2,mGem: A perfect storm in the era of global war...,,False,,,review
8,-2,Predicting biomass global warming potential wi...,PMC12484551,True,,,review
9,-2,Who's Interested in Global Warming?,PMC12610921,True,,,review



High-score papers with XML:
  0 of 50 rows


,score,title,pmcid,has_xml,location_terms,pollutant_terms,review_status


In [ ]:
# Optional: filter by a term in title or abstract
TERM = "India"
mask = df["title"].str.contains(TERM, case=False, na=False) | df[
    "abstract_snippet"
].str.contains(TERM, case=False, na=False)
display(df.loc[mask, display_cols])

,score,title,pmcid,has_xml,location_terms,pollutant_terms,review_status
1,2,Benchmarking Regional Climate Variability in C...,,False,"india, indian",,review


## 5. Download review artifacts

Download the CSV or JSON to edit `review_status` (`include`, `review`, `exclude`) offline.

In [ ]:
from google.colab import files
import shutil

csv_path = result["review_paths"]["csv"]
files.download(csv_path)

# Optional: zip the entire query directory
zip_base = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
files.download(zip_base)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Iterative query refinement

Use this checklist after each run:

- **Too broad?** Add location or topic terms (e.g. `AND Delhi`)
- **Too narrow?** Remove restrictive terms or broaden synonyms
- **Missing fulltext?** Papers without PMCID cannot download XML; note `has_xml = false` rows

To refine: copy parameters into section 2, set `REVISION_OF` to the previous `QUERY_NAME`, and re-run sections 3–5.

In [ ]:
# Example: refined query (edit QUERY_NAME and QUERY_STRING in section 2, then re-run)
REFINED_QUERY_NAME = "global_warming_delhi"
REFINED_QUERY_STRING = (
    '("global warming" OR temperature rise OR climate change) AND India AND Delhi'
)
print("Set in section 2:")
print(f"  QUERY_NAME = {REFINED_QUERY_NAME!r}")
print(f"  QUERY_STRING = {REFINED_QUERY_STRING!r}")
print(f"  REVISION_OF = {QUERY_NAME!r}")

Set in section 2:
  QUERY_NAME = 'global_warming_delhi'
  QUERY_STRING = '("global warming" OR temperature rise OR climate change) AND India AND Delhi'
  REVISION_OF = 'global_warming'


In [ ]:
# Compare two runs by pmcid/pmid (run after a refined query)
import json

def load_ids(path):
    with open(path, "r", encoding="utf-8") as handle:
        rows = json.load(handle)
    ids = set()
    for row in rows:
        key = row.get("pmcid") or row.get("pmid") or row.get("paper_id")
        if key:
            ids.add(key)
    return ids

prior_json = Path("/content/queries", QUERY_NAME if REVISION_OF is None else REVISION_OF, "review", "review_table.json")
current_json = Path(result["review_paths"]["json"])

if prior_json.is_file() and current_json.is_file():
    prior_ids = load_ids(prior_json)
    current_ids = load_ids(current_json)
    overlap = prior_ids & current_ids
    print(f"Prior run: {len(prior_ids)} papers")
    print(f"Current run: {len(current_ids)} papers")
    print(f"Overlap: {len(overlap)} papers")
    print(f"New in current run: {len(current_ids - prior_ids)}")
    print(f"Dropped from prior run: {len(prior_ids - current_ids)}")
else:
    print("Run a refined query first, or adjust prior_json path.")

Prior run: 37 papers
Current run: 37 papers
Overlap: 37 papers
New in current run: 0
Dropped from prior run: 0


## 7. Optional: persist to Google Drive

Colab sessions are ephemeral. Mount Drive and set `OUTPUT_DIR` under your Drive folder to keep results between sessions.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Example: OUTPUT_DIR = Path("/content/drive/MyDrive/semantic_corpus/queries", QUERY_NAME)

Mounted at /content/drive


## 8. Next steps (outside this notebook)

1. Edit `review_status` in the downloaded CSV or JSON (`include` / `exclude`)
2. Ingest selected papers into a BAGIT corpus locally
3. Export a chatbot manifest for RAG indexing

See the [chatbot export contract](https://github.com/semanticClimate/semantic_corpus/blob/main/docs/chatbot_export_contract.md).